In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = 'najwasahidahkautsar'
os.environ['KAGGLE_KEY'] = 'KGAT_ef0bf10010619e3eb2b80cf1a939ef2b'

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

In [ ]:
import zipfile

with zipfile.ZipFile("chest-xray-pneumonia.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

normal_path = "data/chest_xray/train/NORMAL"
pneumonia_path = "data/chest_xray/train/PNEUMONIA"

normal_img = cv2.imread(os.path.join(normal_path, os.listdir(normal_path)[0]))
pneumonia_img = cv2.imread(os.path.join(pneumonia_path, os.listdir(pneumonia_path)[0]))

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.imshow(normal_img)
plt.title("Normal")

plt.subplot(1,2,2)
plt.imshow(pneumonia_img)
plt.title("Pneumonia")

plt.show()

# **Resize**

In [ ]:
img_size = 224

#**ImageDataGenerator**

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = "data/chest_xray/train"
val_dir = "data/chest_xray/val"
test_dir = "data/chest_xray/test"

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

#**Load Data to Model**

In [ ]:
train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

test_data = test_gen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

#**Build CNN**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential()

#Conv Block 1,
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)))
model.add(MaxPooling2D(2,2))

#Conv Block 2,
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

#Conv Block 3,
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

#Flatten,
model.add(Flatten())

#Dense,
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

#Output (binary),
model.add(Dense(1, activation='sigmoid'))

model.summary()

In [ ]:
model.compile(
    optimizer='adam' ,
    loss='binary_crossentropy' ,
    metrics=['accuracy']
)

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

In [ ]:
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop]
)

# **VGG16 transfer learning**

In [ ]:
from tensorflow.keras.applications import VGG16

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

In [ ]:
for layer in base_model.layers:
    layer.trainable = False


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout

model = Sequential([
    base_model,
    Flatten(),
    Dense(224, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

In [ ]:
#Unfreeze half layer

for layer in base_model.layers[:-4] : #freeze for the most part
  layer.trainable = False

for layer in base_model.layers[-4:] : #open last layer
  layer.trainable = True

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop]
)

In [ ]:
test_loss, test_acc = model.evaluate(test_data)
print("Test Accuracy:", test_acc)

# **Confusion Matrix & Classification Report**

In [ ]:
import numpy as np

y_pred = model.predict(test_data)
y_pred = (y_pred > 0.5).astype(int).flatten()

In [ ]:
y_true = test_data.classes

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

In [ ]:
test_data = test_gen.flow_from_directory(
    test_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

In [ ]:
test_data.reset()

In [ ]:
y_pred = model.predict(test_data)
y_pred = (y_pred > 0.5).astype(int).flatten()

y_true = test_data.classes

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d')

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.savefig("confusion_matrix.png")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'])

plt.savefig("accuracy_plot.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'])

plt.savefig("loss_plot.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np

images, labels = next(test_data)

predictions = model.predict(images)
predictions = (predictions > 0.5).astype(int).flatten()

In [ ]:
import matplotlib.pyplot as plt

class_names = ['Normal', 'Pneumonia']

plt.figure(figsize=(10,6))

for i in range(6):
    plt.subplot(2,3,i+1)
    plt.imshow(images[i])

    true_label = class_names[int(labels[i])]
    pred_label = class_names[int(predictions[i])]

    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis('off')

plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np

images, labels = next(test_data)

img = images[0]
label = labels[0]

In [ ]:
img_array = np.expand_dims(img, axis=0)

In [ ]:
last_conv_layer_name = "block5_conv3"

In [ ]:
import tensorflow as tf

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)

    return heatmap.numpy()

In [ ]:
for layer in model.layers:
  print(layer.name)

In [ ]:
base_model = model.get_layer('vgg16')

In [ ]:
images, labels = next(test_data)

img = images[0]
img_array = np.expand_dims(img, axis=0)

In [ ]:
model.predict(img_array)

In [ ]:
inputs = tf.keras.Input(shape=(224,224, 3))

x = model.layers[0](inputs)
x = model.layers[1](x)
x = model.layers[2](x)
x = model.layers[3](x)
outputs = model.layers[4](x)

functional_model = tf.keras.Model(inputs, outputs)

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import Model

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))

last_conv_layer = base_model.get_layer("block5_conv3")

x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

grad_model = Model(inputs=base_model.input, outputs=[last_conv_layer.output, output])

In [ ]:
with tf.GradientTape() as tape:
    conv_outputs, predictions = grad_model(img_array)
    loss = predictions[:, 0]

In [ ]:
heatmap = make_gradcam_heatmap(img_array, base_model, last_conv_layer_name)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(heatmap, cmap='jet')
plt.colorbar()
plt.title("Grad-CAM Heatmap")
plt.show()

In [ ]:
heatmap = np.maximum(heatmap, 0)
heatmap = heatmap / np.max(heatmap)

heatmap_resized = cv2.resize(heatmap, (224, 224))
heatmap_resized = np.uint8(255 * heatmap_resized)

heatmap_color = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

img_original = img.copy()
if img_original.max() <= 1:
    img_original = (img_original * 255).astype("uint8")

superimposed_img = cv2.addWeighted(img_original, 0.6, heatmap_color, 0.4, 0)

plt.imshow(superimposed_img)
plt.axis('off')
plt.title("Grad-CAM Overlay")
plt.show()

**Grad-Cam Comparison: Normal vs Pneumonia**

In [ ]:
print(model.get_layer("vgg16").get_layer("block5_conv3"))

In [ ]:
model.summary()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

def get_gradcam_results(model, test_data):
    for images, labels in test_data:
        if (0 in labels) and (1 in labels):
            break

    normal_idx = np.where(labels == 0)[0][0]
    pneumonia_idx = np.where(labels == 1)[0][0]

    samples = [
        (images[normal_idx], "Normal"),
        (images[pneumonia_idx], "Pneumonia")
    ]

    vgg_base = model.get_layer('vgg16')
    post_vgg_layers = model.layers[1:]

    grad_model = tf.keras.models.Model(
        inputs=vgg_base.input,
        outputs=[
            vgg_base.get_layer("block5_conv3").output,
            vgg_base.output
        ]
    )

    results = []

    for img, title in samples:
        img_array = tf.cast(np.expand_dims(img, axis=0), tf.float32)

        with tf.GradientTape() as tape:
            conv_outputs, vgg_out = grad_model(img_array)

            tape.watch(conv_outputs)

            x = vgg_out
            for layer in post_vgg_layers:
                x = layer(x)

            loss = x[:, 0]

        grads = tape.gradient(loss, conv_outputs)
        if grads is None:
          print("Gradient none untuk {title}, skkip")
          continue

        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

        heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap).numpy()
        heatmap = np.maximum(heatmap, 0)
        heatmap /= (np.max(heatmap) + 1e-10)

        heatmap_resized = cv2.resize(heatmap, (224, 224))
        heatmap_color = cv2.applyColorMap(
            np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
        )
        heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

        img_8bit = np.uint8(255 * img) if img.max() <= 1.0 else np.uint8(img)

        if len(img_8bit.shape) == 2:
          img_8bit = cv2.cvtColor(img_8bit, cv2.COLOR_GRAY2RGB)

        overlay = cv2.addWeighted(img_8bit, 0.6, heatmap_color, 0.4, 0)

        results.append((img, overlay, title))

    return results

hasil = get_gradcam_results(model, test_data)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for i, (original, cam, title) in enumerate(hasil):
    axes[0, i].imshow(original)
    axes[0, i].set_title(f"{title} - Original")
    axes[0, i].axis("off")

    axes[1, i].imshow(cam)
    axes[1, i].set_title(f"{title} - Grad-CAM Area")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()